In [0]:


-- 1. Parse streaming Kafka Inventory and extract JSON payload
CREATE STREAMING LIVE VIEW stg_kafka_inventory AS
SELECT
  from_json(
    message_payload, 
    'event_id STRING, restock_date STRING, product_id STRING, store_id STRING, restock_quantity STRING'
  ) AS payload
FROM STREAM(mavenmarket.bronze.kafka_inventory);

-- 2. Normalize dates and cast data types
CREATE STREAMING LIVE VIEW stg_inventory_cleaned AS
SELECT
  payload.event_id,
  -- Handle date formatting anomalies (MM/DD/YYYY vs YYYY-MM-DD)
  COALESCE(
    try_to_date(payload.restock_date, 'MM/dd/yyyy'),
    try_to_date(payload.restock_date, 'yyyy-MM-dd')
  ) AS restock_date,
  CAST(payload.product_id AS INT) AS product_id,
  CAST(payload.store_id AS INT) AS store_id,
  CAST(payload.restock_quantity AS INT) AS restock_quantity
FROM STREAM(live.stg_kafka_inventory);

-- 3. The Clean Silver Table with Native DLT Expectations
CREATE STREAMING LIVE TABLE silver_inventory (
  CONSTRAINT valid_restock EXPECT (restock_quantity > 0) ON VIOLATION DROP ROW,
  CONSTRAINT valid_store EXPECT (store_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_date EXPECT (restock_date IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT "Cleaned near-real-time inventory restocks"
AS SELECT * FROM STREAM(live.stg_inventory_cleaned);